In [14]:
from pathlib import Path

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")

PARENT_MODEL_PATH = BUILD_DIR / "intermediate_models" / "dataflow_parent.onnx"
DEPLOY_DIR_PC = BUILD_DIR / "deploy"

# 优先选最接近板子的 child
child_candidates = [
    BUILD_DIR / "intermediate_models" / "step_set_fifo_depths.onnx",
    BUILD_DIR / "intermediate_models" / "step_hw_ipgen.onnx",
    BUILD_DIR / "intermediate_models" / "step_hw_codegen.onnx",
]

CHILD_MODEL_PATH = None
for p in child_candidates:
    if p.exists():
        CHILD_MODEL_PATH = p
        break

if CHILD_MODEL_PATH is None:
    raise FileNotFoundError("No suitable child model found in intermediate_models")

print("Using child model:", CHILD_MODEL_PATH)
print("Using parent     :", PARENT_MODEL_PATH)
print("Using deploy dir :", DEPLOY_DIR_PC)

Using child model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx
Using parent     : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/dataflow_parent.onnx
Using deploy dir : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/deploy


In [1]:
from pathlib import Path
import json
import numpy as np
import onnx
from onnx import numpy_helper

# =========================
# 电脑端配置
# =========================
PARENT_ONNX = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/dataflow_parent.onnx")
DATASET_TXT = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt")

OUT_DIR = Path("./board_split_io")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SAMPLES = None      # None = 全量
SHUFFLE = False
SEED = 0
LABEL_OFFSET = 1        # ECG5000: 1~5 -> 0~4
MT_CMP = ">="           # 通常先用 >=


def get_initializer_dict(model):
    return {init.name: numpy_helper.to_array(init) for init in model.graph.initializer}


def get_attr(node, name, default=None):
    for a in node.attribute:
        if a.name == name:
            return onnx.helper.get_attribute_value(a)
    return default


def parse_parent_sw(parent_onnx_path):
    model = onnx.load(str(parent_onnx_path))
    init_dict = get_initializer_dict(model)
    nodes = list(model.graph.node)

    sdp_idx = None
    for i, n in enumerate(nodes):
        if n.op_type == "StreamingDataflowPartition":
            sdp_idx = i
            break
    if sdp_idx is None:
        raise RuntimeError("没有找到 StreamingDataflowPartition")

    head_nodes = nodes[:sdp_idx]
    tail_nodes = nodes[sdp_idx + 1:]

    mt_node = None
    tr0_node = None
    tr1_node = None
    mul_node = None
    add_node = None

    for n in head_nodes:
        if n.op_type == "MultiThreshold" and mt_node is None:
            mt_node = n
        elif n.op_type == "Transpose" and tr0_node is None:
            tr0_node = n

    for n in tail_nodes:
        if n.op_type == "Transpose" and tr1_node is None:
            tr1_node = n
        elif n.op_type == "Mul" and mul_node is None:
            mul_node = n
        elif n.op_type == "Add" and add_node is None:
            add_node = n

    if mt_node is None or tr0_node is None or tr1_node is None or mul_node is None or add_node is None:
        raise RuntimeError("没有完整解析出 SW 首尾节点")

    mt_thresholds = init_dict[mt_node.input[1]].astype(np.float32)
    mt_out_scale = float(get_attr(mt_node, "out_scale", 1.0))
    mt_out_bias = float(get_attr(mt_node, "out_bias", 0.0))
    tr0_perm = tuple(get_attr(tr0_node, "perm"))
    tr1_perm = tuple(get_attr(tr1_node, "perm"))

    mul_const = None
    for name in mul_node.input:
        if name in init_dict:
            mul_const = init_dict[name].astype(np.float32)
            break
    if mul_const is None:
        raise RuntimeError("没有找到 Mul 常量")

    add_const = None
    for name in add_node.input:
        if name in init_dict:
            add_const = init_dict[name].astype(np.float32)
            break
    if add_const is None:
        raise RuntimeError("没有找到 Add 常量")

    return {
        "mt_thresholds": mt_thresholds,
        "mt_out_scale": mt_out_scale,
        "mt_out_bias": mt_out_bias,
        "tr0_perm": tr0_perm,
        "tr1_perm": tr1_perm,
        "mul_const": mul_const,
        "add_const": add_const,
    }


def multithreshold_nchw(x, thresholds, out_scale=1.0, out_bias=0.0, cmp_mode=">="):
    x = np.asarray(x, dtype=np.float32)
    th = np.asarray(thresholds, dtype=np.float32)

    if x.ndim != 4:
        raise ValueError(f"Expected [N,C,H,W], got {x.shape}")

    n, c, h, w = x.shape

    if th.ndim == 1:
        th = th.reshape(1, -1)
    if th.shape[0] == 1 and c > 1:
        th = np.repeat(th, c, axis=0)
    if th.shape[0] != c:
        raise ValueError(f"Threshold channels {th.shape[0]} != input channels {c}")

    y = np.empty_like(x, dtype=np.float32)

    for ch in range(c):
        xc = x[:, ch, :, :]
        tc = th[ch].reshape(1, -1, 1, 1)
        xce = xc[:, None, :, :]

        if cmp_mode == ">=":
            cnt = (xce >= tc).sum(axis=1)
        elif cmp_mode == ">":
            cnt = (xce > tc).sum(axis=1)
        else:
            raise ValueError(f"Unsupported cmp_mode: {cmp_mode}")

        y[:, ch, :, :] = cnt.astype(np.float32)

    y = y * out_scale + out_bias
    return y.astype(np.float32)


def sw_head(x_raw, meta):
    x = multithreshold_nchw(
        x_raw,
        meta["mt_thresholds"],
        out_scale=meta["mt_out_scale"],
        out_bias=meta["mt_out_bias"],
        cmp_mode=MT_CMP,
    )
    x = np.transpose(x, meta["tr0_perm"]).astype(np.float32)
    return x


def load_ecg5000_txt(path, max_samples=None, shuffle=False, seed=0):
    arr = np.loadtxt(str(path), dtype=np.float32)
    y = arr[:, 0].astype(np.int64) - LABEL_OFFSET
    x = arr[:, 1:].astype(np.float32)

    idx = np.arange(len(x))
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
    if max_samples is not None:
        idx = idx[:max_samples]

    return x[idx], y[idx], idx


def main():
    meta = parse_parent_sw(PARENT_ONNX)
    x_flat, y, orig_idx = load_ecg5000_txt(DATASET_TXT, MAX_SAMPLES, SHUFFLE, SEED)

    # [N, 140] -> [N, 1, 1, 140]
    x_raw_all = x_flat[:, None, None, :].astype(np.float32)
    x_hw_all = sw_head(x_raw_all, meta)

    np.save(OUT_DIR / "x_hw_all.npy", x_hw_all)
    np.save(OUT_DIR / "labels.npy", y)
    np.save(OUT_DIR / "orig_indices.npy", orig_idx)

    # 尾部补偿参数给电脑端 finalize 用
    np.savez(
        OUT_DIR / "sw_tail_meta.npz",
        tr1_perm=np.array(meta["tr1_perm"], dtype=np.int64),
        mul_const=meta["mul_const"].astype(np.float32),
        add_const=meta["add_const"].astype(np.float32),
    )

    info = {
        "num_samples": int(len(y)),
        "raw_input_shape": list(x_raw_all.shape),
        "hw_input_shape": list(x_hw_all.shape),
        "labels_unique": [int(v) for v in np.unique(y)],
        "mt_cmp": MT_CMP,
    }
    with open(OUT_DIR / "prepare_info.json", "w") as f:
        json.dump(info, f, indent=2)

    print("[done] saved:")
    print(" ", OUT_DIR / "x_hw_all.npy")
    print(" ", OUT_DIR / "labels.npy")
    print(" ", OUT_DIR / "orig_indices.npy")
    print(" ", OUT_DIR / "sw_tail_meta.npz")
    print(" ", OUT_DIR / "prepare_info.json")
    print("[info] hw input shape:", x_hw_all.shape)


if __name__ == "__main__":
    main()

[done] saved:
  board_split_io/x_hw_all.npy
  board_split_io/labels.npy
  board_split_io/orig_indices.npy
  board_split_io/sw_tail_meta.npz
  board_split_io/prepare_info.json
[info] hw input shape: (4500, 1, 140, 1)


In [2]:
from pathlib import Path
import csv
import json
import numpy as np

# =========================
# 电脑端配置
# =========================
IO_DIR = Path("./board_split_io")

Y_HW_NPY = IO_DIR / "y_hw_all.npy"
LABELS_NPY = IO_DIR / "labels.npy"
ORIG_IDX_NPY = IO_DIR / "orig_indices.npy"
SW_TAIL_META_NPZ = IO_DIR / "sw_tail_meta.npz"

SAVE_ROWS_CSV = True
SAVE_SUMMARY_JSON = True


def sw_tail(y_hw, meta):
    y = np.asarray(y_hw, dtype=np.float32)
    y = np.transpose(y, tuple(meta["tr1_perm"].tolist())).astype(np.float32)
    y = y * meta["mul_const"]
    y = y + meta["add_const"]
    return y.astype(np.float32)


def normalize_logits(y):
    y = np.asarray(y, dtype=np.float32)
    if y.ndim == 4 and y.shape[2] == 1 and y.shape[3] == 1:
        y = y[:, :, 0, 0]
    elif y.ndim != 2:
        y = y.reshape(y.shape[0], -1)
    return y


def main():
    meta = np.load(SW_TAIL_META_NPZ)
    y_hw_all = np.load(Y_HW_NPY).astype(np.float32)
    labels = np.load(LABELS_NPY).astype(np.int64)
    orig_idx = np.load(ORIG_IDX_NPY).astype(np.int64)

    y_final_all = sw_tail(y_hw_all, meta)
    logits_2d = normalize_logits(y_final_all)
    preds = logits_2d.argmax(axis=1)

    acc = float((preds == labels).mean())

    class_total = {}
    class_correct = {}
    for yt, yp in zip(labels, preds):
        yt = int(yt)
        yp = int(yp)
        class_total[yt] = class_total.get(yt, 0) + 1
        class_correct[yt] = class_correct.get(yt, 0) + int(yt == yp)

    per_class_acc = {}
    for c in sorted(class_total.keys()):
        per_class_acc[c] = class_correct[c] / class_total[c]

    print("\n========== board finalize summary ==========")
    print("num_samples:", len(labels))
    print("accuracy:", acc)
    print("per_class_accuracy:", per_class_acc)

    np.save(IO_DIR / "y_final_all.npy", y_final_all)
    np.save(IO_DIR / "preds.npy", preds)

    rows = []
    for i in range(len(labels)):
        rows.append({
            "local_index": i,
            "orig_index": int(orig_idx[i]),
            "label": int(labels[i]),
            "pred": int(preds[i]),
            "correct": int(preds[i] == labels[i]),
        })

    if SAVE_ROWS_CSV:
        csv_path = IO_DIR / "final_rows.csv"
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        print("[info] saved rows csv:", csv_path)

    if SAVE_SUMMARY_JSON:
        summary = {
            "num_samples": int(len(labels)),
            "accuracy": acc,
            "per_class_accuracy": per_class_acc,
        }
        json_path = IO_DIR / "final_summary.json"
        with open(json_path, "w") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        print("[info] saved summary json:", json_path)


if __name__ == "__main__":
    main()


========== board finalize summary ==========
num_samples: 4500
accuracy: 0.004888888888888889
per_class_accuracy: {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 1.0}
[info] saved rows csv: board_split_io/final_rows.csv
[info] saved summary json: board_split_io/final_summary.json


In [3]:
import numpy as np

x_hw = np.load("./board_split_io/x_hw_all.npy")
print("shape:", x_hw.shape)
print("min :", x_hw.min())
print("max :", x_hw.max())
print("mean:", x_hw.mean())
print("unique first sample:", np.unique(x_hw[0])[:50])

shape: (4500, 1, 140, 1)
min : -113.0
max : 118.0
mean: -0.00026507938
unique first sample: [-73. -66. -55. -34. -31. -30. -22. -21. -20. -19. -18. -17. -16. -11.
 -10.  -9.  -8.  -7.  -6.  -5.  -3.  -2.  -1.   0.   1.   2.   3.   4.
   5.   6.   7.   8.   9.  10.  11.  12.  13.  14.  15.  17.  19.  21.
  24.  27.  59.]


In [15]:
from pathlib import Path
import numpy as np

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode


# =========================================================
# 1. 路径：按你现在这版写死了
# =========================================================
CHILD_MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_hw_ipgen.onnx"

# 这是你电脑端前处理后生成的 bitfile 输入
X_HW_NPY = "./board_split_io/x_hw_all.npy"

# 如果你已经把板子上的原始输出拷回来了，就填这个；否则也可以先只跑电脑端
BOARD_Y_HW_NPY = "./board_split_io/y_hw_all.npy"

OUT_DIR = Path("./board_split_io/pc_child_compare")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 先跑多少条；None 表示全量
MAX_SAMPLES = 20

# 是否保存电脑端 raw child 输出
SAVE_PC_RAW = True


# =========================================================
# 2. 把 step_hw_codegen.onnx 变成可执行 cppsim child
# =========================================================
def prepare_child_cppsim(child_model_path, out_dir):
    model = ModelWrapper(child_model_path)
    model = model.transform(PrepareCppSim())
    model = model.transform(CompileCppSim())
    model = model.transform(SetExecMode("cppsim"))

    out_path = str(Path(out_dir) / "step_hw_codegen_cppsim.onnx")
    model.save(out_path)
    print("[info] saved cppsim child model to:", out_path)
    return out_path


# =========================================================
# 3. 运行电脑端 raw child
# =========================================================
def run_pc_child_raw(child_model_path, x_hw_all, max_samples=None):
    model = ModelWrapper(child_model_path)
    inp_name = model.graph.input[0].name
    out_name = model.graph.output[0].name

    n = len(x_hw_all)
    if max_samples is not None:
        n = min(n, max_samples)

    ys = []
    for i in range(n):
        x = x_hw_all[i:i+1].astype(np.float32)
        out_dict = oxe.execute_onnx(model, {inp_name: x}, True)
        y = out_dict[out_name]
        ys.append(np.asarray(y, dtype=np.float32))

        if (i + 1) % 10 == 0 or (i + 1) == n:
            print(f"[progress] pc child {i+1}/{n}")

    y_all = np.concatenate(ys, axis=0)
    return y_all


# =========================================================
# 4. 对比工具
# =========================================================
def summarize_arr(name, arr):
    arr = np.asarray(arr)
    print(f"\n[{name}]")
    print("  shape:", arr.shape)
    print("  dtype:", arr.dtype)
    print("  min  :", arr.min())
    print("  max  :", arr.max())
    print("  mean :", arr.mean())
    print("  sample 0:", arr[0])
    if len(arr) > 1:
        print("  sample 1:", arr[1])
    if len(arr) > 2:
        print("  sample 2:", arr[2])


def compare_pc_vs_board(y_pc, y_board):
    y_pc = np.asarray(y_pc, dtype=np.float32)
    y_board = np.asarray(y_board, dtype=np.float32)

    print("\n========== compare pc child raw vs board raw ==========")
    print("pc    shape:", y_pc.shape)
    print("board shape:", y_board.shape)

    if y_pc.shape != y_board.shape:
        print("[warn] shape mismatch, can't do elementwise compare")
        return

    diff = np.abs(y_pc.astype(np.float64) - y_board.astype(np.float64))
    print("max abs diff :", diff.max())
    print("mean abs diff:", diff.mean())

    pc_pred = y_pc.reshape(y_pc.shape[0], -1).argmax(axis=1)
    bd_pred = y_board.reshape(y_board.shape[0], -1).argmax(axis=1)
    print("pred agreement:", np.mean(pc_pred == bd_pred))

    # 打印前几个不一致样本
    mismatch = np.where(pc_pred != bd_pred)[0]
    print("num pred mismatch:", len(mismatch))
    if len(mismatch) > 0:
        print("first mismatch indices:", mismatch[:20])


# =========================================================
# 5. 主流程
# =========================================================
def main():
    x_hw_all = np.load(X_HW_NPY).astype(np.float32)
    print("[info] loaded x_hw_all:", x_hw_all.shape)

    cppsim_child = prepare_child_cppsim(CHILD_MODEL_PATH, OUT_DIR)

    y_pc = run_pc_child_raw(cppsim_child, x_hw_all, max_samples=MAX_SAMPLES)
    summarize_arr("pc_child_raw", y_pc)

    if SAVE_PC_RAW:
        np.save(OUT_DIR / "y_pc_child_raw.npy", y_pc)
        print("[info] saved pc raw child output to:", OUT_DIR / "y_pc_child_raw.npy")

    board_path = Path(BOARD_Y_HW_NPY)
    if board_path.exists():
        y_board = np.load(board_path).astype(np.float32)

        if MAX_SAMPLES is not None:
            y_board = y_board[:len(y_pc)]

        summarize_arr("board_child_raw", y_board)
        compare_pc_vs_board(y_pc, y_board)
    else:
        print("[info] board y_hw_all.npy not found, skipped board comparison")


if __name__ == "__main__":
    main()

[info] loaded x_hw_all: (4500, 1, 140, 1)
[info] saved cppsim child model to: board_split_io/pc_child_compare/step_hw_codegen_cppsim.onnx
[progress] pc child 10/20
[progress] pc child 20/20

[pc_child_raw]
  shape: (20, 1, 1, 5)
  dtype: float32
  min  : -916658.0
  max  : 530692.0
  mean : -483421.25
  sample 0: [[[ 430937. -735194. -681037. -818776. -629093.]]]
  sample 1: [[[ 460439. -650723. -844528. -721681. -633197.]]]
  sample 2: [[[ 232264. -489421. -751060. -596628. -825782.]]]
[info] saved pc raw child output to: board_split_io/pc_child_compare/y_pc_child_raw.npy

[board_child_raw]
  shape: (20, 1, 1, 5)
  dtype: float32
  min  : 0.0
  max  : 3.0
  mean : 1.45
  sample 0: [[[1. 2. 3. 0. 3.]]]
  sample 1: [[[3. 1. 0. 3. 3.]]]
  sample 2: [[[0. 3. 0. 0. 2.]]]

========== compare pc child raw vs board raw ==========
pc    shape: (20, 1, 1, 5)
board shape: (20, 1, 1, 5)
max abs diff : 916660.0
mean abs diff: 626005.68
pred agreement: 0.3
num pred mismatch: 14
first mismatch indic

raw board y_hw accuracy: 0.29288888888888887
unique preds: (array([0, 1, 2, 3, 4]), array([1529, 1131,  823,  602,  415]))


In [17]:
from pathlib import Path
import numpy as np

from qonnx.core.modelwrapper import ModelWrapper
#import qonnx.core.onnx_exec as oxe
from finn.core.onnx_exec import execute_onnx as finn_execute_onnx

from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode


# =========================================================
# 1. 路径：按你现在这版写死了
# =========================================================
#from pathlib import Path

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
INTER_DIR = BUILD_DIR / "intermediate_models"

preferred = [
    INTER_DIR / "step_set_fifo_depths.onnx",
    INTER_DIR / "step_hw_ipgen.onnx",
]

CHILD_MODEL_PATH = None
for p in preferred:
    if p.exists():
        CHILD_MODEL_PATH = p
        break

if CHILD_MODEL_PATH is None:
    raise FileNotFoundError("没有找到 step_set_fifo_depths.onnx 或 step_hw_ipgen.onnx")

print("[info] using child model:", CHILD_MODEL_PATH)
# 这是你电脑端前处理后生成的 bitfile 输入
X_HW_NPY = "./board_split_io/x_hw_all.npy"

# 如果你已经把板子上的原始输出拷回来了，就填这个；否则也可以先只跑电脑端
BOARD_Y_HW_NPY = "./board_split_io/y_hw_all.npy"

OUT_DIR = Path("./board_split_io/pc_child_compare")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 先跑多少条；None 表示全量
MAX_SAMPLES = 20

# 是否保存电脑端 raw child 输出
SAVE_PC_RAW = True


# =========================================================
# 2. 把 step_hw_codegen.onnx 变成可执行 cppsim child
# =========================================================
def prepare_child_cppsim(child_model_path, out_dir):
    model = ModelWrapper(child_model_path)
    model = model.transform(PrepareCppSim())
    model = model.transform(CompileCppSim())
    model = model.transform(SetExecMode("cppsim"))

    out_path = str(Path(out_dir) / "step_hw_codegen_cppsim.onnx")
    model.save(out_path)
    print("[info] saved cppsim child model to:", out_path)
    return out_path


# =========================================================
# 3. 运行电脑端 raw child
# =========================================================
def run_pc_child_raw(child_model_path, x_hw_all, max_samples=None):
    model = ModelWrapper(child_model_path)
    inp_name = model.graph.input[0].name
    out_name = model.graph.output[0].name

    n = len(x_hw_all)
    if max_samples is not None:
        n = min(n, max_samples)

    ys = []
    for i in range(n):
        x = x_hw_all[i:i+1].astype(np.float32)
        out_dict = oxe.execute_onnx(model, {inp_name: x}, True)
        y = out_dict[out_name]
        ys.append(np.asarray(y, dtype=np.float32))

        if (i + 1) % 10 == 0 or (i + 1) == n:
            print(f"[progress] pc child {i+1}/{n}")

    y_all = np.concatenate(ys, axis=0)
    return y_all


# =========================================================
# 4. 对比工具
# =========================================================
def summarize_arr(name, arr):
    arr = np.asarray(arr)
    print(f"\n[{name}]")
    print("  shape:", arr.shape)
    print("  dtype:", arr.dtype)
    print("  min  :", arr.min())
    print("  max  :", arr.max())
    print("  mean :", arr.mean())
    print("  sample 0:", arr[0])
    if len(arr) > 1:
        print("  sample 1:", arr[1])
    if len(arr) > 2:
        print("  sample 2:", arr[2])


def compare_pc_vs_board(y_pc, y_board):
    y_pc = np.asarray(y_pc, dtype=np.float32)
    y_board = np.asarray(y_board, dtype=np.float32)

    print("\n========== compare pc child raw vs board raw ==========")
    print("pc    shape:", y_pc.shape)
    print("board shape:", y_board.shape)

    if y_pc.shape != y_board.shape:
        print("[warn] shape mismatch, can't do elementwise compare")
        return

    diff = np.abs(y_pc.astype(np.float64) - y_board.astype(np.float64))
    print("max abs diff :", diff.max())
    print("mean abs diff:", diff.mean())

    pc_pred = y_pc.reshape(y_pc.shape[0], -1).argmax(axis=1)
    bd_pred = y_board.reshape(y_board.shape[0], -1).argmax(axis=1)
    print("pred agreement:", np.mean(pc_pred == bd_pred))

    # 打印前几个不一致样本
    mismatch = np.where(pc_pred != bd_pred)[0]
    print("num pred mismatch:", len(mismatch))
    if len(mismatch) > 0:
        print("first mismatch indices:", mismatch[:20])


# =========================================================
# 5. 主流程
# =========================================================
def main():
    x_hw_all = np.load(X_HW_NPY).astype(np.float32)
    print("[info] loaded x_hw_all:", x_hw_all.shape)

    cppsim_child = prepare_child_cppsim(CHILD_MODEL_PATH, OUT_DIR)

    y_pc = run_pc_child_raw(cppsim_child, x_hw_all, max_samples=MAX_SAMPLES)
    summarize_arr("pc_child_raw", y_pc)

    if SAVE_PC_RAW:
        np.save(OUT_DIR / "y_pc_child_raw.npy", y_pc)
        print("[info] saved pc raw child output to:", OUT_DIR / "y_pc_child_raw.npy")

    board_path = Path(BOARD_Y_HW_NPY)
    if board_path.exists():
        y_board = np.load(board_path).astype(np.float32)

        if MAX_SAMPLES is not None:
            y_board = y_board[:len(y_pc)]

        summarize_arr("board_child_raw", y_board)
        compare_pc_vs_board(y_pc, y_board)
    else:
        print("[info] board y_hw_all.npy not found, skipped board comparison")


if __name__ == "__main__":
    main()

[info] using child model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx
[info] loaded x_hw_all: (4500, 1, 140, 1)


AttributeError: 'PosixPath' object has no attribute 'graph'

In [3]:
from pathlib import Path
import json
import numpy as np
import onnx
from onnx import numpy_helper

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import qonnx.core.onnx_exec as qoxe
from finn.core.onnx_exec import execute_onnx as finn_execute_onnx

from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode

# =========================================================
# 固定路径：按你当前环境写
# =========================================================
BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
INTER_DIR = BUILD_DIR / "intermediate_models"
PARENT_ONNX = INTER_DIR / "dataflow_parent.onnx"

IO_DIR = Path("./board_split_io")
X_HW_NPY = IO_DIR / "x_hw_all.npy"
Y_BOARD_RAW_NPY = IO_DIR / "y_hw_all.npy"
LABELS_NPY = IO_DIR / "labels.npy"

OUT_DIR = IO_DIR / "debug_mismatch_rootcause"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 先看前多少条
MAX_SAMPLES = 50

# 打印前几个 mismatch
NUM_PRINT_MISMATCH = 10

# 自动选更接近板子的 child
CHILD_CANDIDATES = [
    INTER_DIR / "step_set_fifo_depths.onnx",
    INTER_DIR / "step_hw_ipgen.onnx",
    INTER_DIR / "step_hw_codegen.onnx",
]

# =========================================================
# 工具
# =========================================================
def choose_child():
    for p in CHILD_CANDIDATES:
        if p.exists():
            return p
    raise FileNotFoundError("No suitable child model found")


def get_initializer_dict(model):
    return {init.name: numpy_helper.to_array(init) for init in model.graph.initializer}


def get_attr(node, name, default=None):
    for a in node.attribute:
        if a.name == name:
            return onnx.helper.get_attribute_value(a)
    return default


def normalize_logits(y):
    y = np.asarray(y, dtype=np.float32)
    if y.ndim == 4 and y.shape[2] == 1 and y.shape[3] == 1:
        y = y[:, :, 0, 0]
    elif y.ndim == 2:
        pass
    else:
        y = y.reshape(y.shape[0], -1)
    return y


def pred_from_logits(y):
    y2 = normalize_logits(y)
    return y2.argmax(axis=1)


def summarize(name, arr):
    arr = np.asarray(arr)
    print(f"\n[{name}]")
    print("  shape:", arr.shape)
    print("  dtype:", arr.dtype)
    print("  min  :", arr.min())
    print("  max  :", arr.max())
    print("  mean :", arr.mean())
    if len(arr) > 0:
        print("  sample0:", arr[0])
    if len(arr) > 1:
        print("  sample1:", arr[1])
    if len(arr) > 2:
        print("  sample2:", arr[2])


# =========================================================
# 解析 parent 的软件尾部
# =========================================================
def parse_parent_tail(parent_onnx_path):
    model = onnx.load(str(parent_onnx_path))
    init_dict = get_initializer_dict(model)
    nodes = list(model.graph.node)

    sdp_idx = None
    for i, n in enumerate(nodes):
        if n.op_type == "StreamingDataflowPartition":
            sdp_idx = i
            break
    if sdp_idx is None:
        raise RuntimeError("No StreamingDataflowPartition found")

    tail_nodes = nodes[sdp_idx + 1 :]

    tr1_node = None
    mul_node = None
    add_node = None

    for n in tail_nodes:
        if n.op_type == "Transpose" and tr1_node is None:
            tr1_node = n
        elif n.op_type == "Mul" and mul_node is None:
            mul_node = n
        elif n.op_type == "Add" and add_node is None:
            add_node = n

    if tr1_node is None or mul_node is None or add_node is None:
        raise RuntimeError("Tail SW ops not fully found")

    tr1_perm = tuple(get_attr(tr1_node, "perm"))

    mul_const = None
    for name in mul_node.input:
        if name in init_dict:
            mul_const = init_dict[name].astype(np.float32)
            break
    if mul_const is None:
        raise RuntimeError("Mul const not found")

    add_const = None
    for name in add_node.input:
        if name in init_dict:
            add_const = init_dict[name].astype(np.float32)
            break
    if add_const is None:
        raise RuntimeError("Add const not found")

    return {
        "tr1_perm": tr1_perm,
        "mul_const": mul_const,
        "add_const": add_const,
    }


def sw_tail(y_hw, meta):
    y = np.asarray(y_hw, dtype=np.float32)
    y = np.transpose(y, meta["tr1_perm"]).astype(np.float32)
    y = y * meta["mul_const"]
    y = y + meta["add_const"]
    return y.astype(np.float32)


# =========================================================
# 准备并运行电脑端 child cppsim
# =========================================================
def prepare_child_cppsim(child_model_path, out_dir):
    model = ModelWrapper(str(child_model_path))

    print("[debug] original model exec_mode:", repr(model.get_metadata_prop("exec_mode")))

    # 节点级 cppsim 准备
    model = model.transform(PrepareCppSim())
    model = model.transform(CompileCppSim())
    model = model.transform(SetExecMode("cppsim"))

    # 关键：清掉顶层模型 metadata，避免 execute_onnx 走 whole-model rtlsim/cppsim 分支
    for k in ["exec_mode", "wrapper_filename", "vivado_stitch_proj", "rtlsim_trace"]:
        try:
            model.set_metadata_prop(k, "")
        except Exception:
            pass

    print("[debug] cleaned model exec_mode:", repr(model.get_metadata_prop("exec_mode")))

    out_path = str(Path(out_dir) / f"{Path(child_model_path).stem}_cppsim.onnx")
    model.save(out_path)
    print("[info] saved cppsim child:", out_path)
    return out_path


def run_pc_child_raw(child_cppsim_path, x_hw_all):
    model = ModelWrapper(str(child_cppsim_path))
    inp_name = model.graph.input[0].name
    out_name = model.graph.output[0].name

    print("[debug] execute model exec_mode:", repr(model.get_metadata_prop("exec_mode")))

    ys = []
    n = len(x_hw_all)
    for i in range(n):
        x = x_hw_all[i:i+1].astype(np.float32)
        out_dict = qoxe.execute_onnx(model, {inp_name: x}, True)
        ys.append(np.asarray(out_dict[out_name], dtype=np.float32))

        if (i + 1) % 10 == 0 or (i + 1) == n:
            print(f"[progress] pc child {i+1}/{n}")

    return np.concatenate(ys, axis=0)


# =========================================================
# 主逻辑
# =========================================================
def main():
    child_model = choose_child()
    print("[info] using child model:", child_model)

    x_hw_all = np.load(X_HW_NPY).astype(np.float32)
    y_board_raw = np.load(Y_BOARD_RAW_NPY).astype(np.float32)
    labels = np.load(LABELS_NPY).astype(np.int64)

    if MAX_SAMPLES is not None:
        x_hw_all = x_hw_all[:MAX_SAMPLES]
        y_board_raw = y_board_raw[:MAX_SAMPLES]
        labels = labels[:MAX_SAMPLES]

    summarize("x_hw_all", x_hw_all)
    summarize("board_raw", y_board_raw)

    tail_meta = parse_parent_tail(PARENT_ONNX)
    print("\n[tail meta]")
    print("  tr1_perm :", tail_meta["tr1_perm"])
    print("  mul_const:", tail_meta["mul_const"])
    print("  add_const:", tail_meta["add_const"])

    child_cppsim = prepare_child_cppsim(child_model, OUT_DIR)
    y_pc_raw = run_pc_child_raw(child_cppsim, x_hw_all)
    summarize("pc_raw", y_pc_raw)

    # 直接 raw argmax
    pred_pc_raw = pred_from_logits(y_pc_raw)
    pred_bd_raw = pred_from_logits(y_board_raw)

    acc_pc_raw = np.mean(pred_pc_raw == labels)
    acc_bd_raw = np.mean(pred_bd_raw == labels)
    agree_raw = np.mean(pred_pc_raw == pred_bd_raw)

    # 补软件尾部后的 argmax
    y_pc_final = sw_tail(y_pc_raw, tail_meta)
    y_bd_final = sw_tail(y_board_raw, tail_meta)

    pred_pc_final = pred_from_logits(y_pc_final)
    pred_bd_final = pred_from_logits(y_bd_final)

    acc_pc_final = np.mean(pred_pc_final == labels)
    acc_bd_final = np.mean(pred_bd_final == labels)
    agree_final = np.mean(pred_pc_final == pred_bd_final)

    # 保存
    np.save(OUT_DIR / "y_pc_raw.npy", y_pc_raw)
    np.save(OUT_DIR / "y_pc_final.npy", y_pc_final)
    np.save(OUT_DIR / "y_bd_final.npy", y_bd_final)

    summary = {
        "child_model": str(child_model),
        "num_samples": int(len(labels)),
        "acc_pc_raw": float(acc_pc_raw),
        "acc_bd_raw": float(acc_bd_raw),
        "agree_raw": float(agree_raw),
        "acc_pc_final": float(acc_pc_final),
        "acc_bd_final": float(acc_bd_final),
        "agree_final": float(agree_final),
    }

    print("\n========== summary ==========")
    for k, v in summary.items():
        print(f"{k}: {v}")

    # 分类判断
    print("\n========== diagnosis ==========")
    if acc_pc_raw > 0.8 and acc_bd_raw < 0.5:
        print("board raw 已经和 pc raw 不一致，问题在板子输出语义/板级执行，不在 sw_tail。")
    elif acc_pc_raw < 0.5 and acc_bd_raw < 0.5:
        print("pc raw 和 board raw 都不行，问题更像 child 参考本身选错。")
    elif acc_pc_raw > 0.8 and acc_bd_raw > 0.8 and acc_bd_final < 0.5:
        print("board raw 本身能分类，但 sw_tail 一补就坏，问题在 sw_tail。")
    else:
        print("需要结合 mismatch 样本继续看。")

    # 打印 mismatch 样本
    print("\n========== mismatch samples ==========")
    mismatch = np.where(pred_pc_raw != pred_bd_raw)[0]
    print("num raw mismatches:", len(mismatch))

    for idx in mismatch[:NUM_PRINT_MISMATCH]:
        print(f"\n--- sample {idx} ---")
        print("label        :", int(labels[idx]))
        print("pc_raw       :", normalize_logits(y_pc_raw[idx:idx+1])[0])
        print("board_raw    :", normalize_logits(y_board_raw[idx:idx+1])[0])
        print("pc_raw_pred  :", int(pred_pc_raw[idx]))
        print("bd_raw_pred  :", int(pred_bd_raw[idx]))
        print("pc_final     :", normalize_logits(y_pc_final[idx:idx+1])[0])
        print("board_final  :", normalize_logits(y_bd_final[idx:idx+1])[0])
        print("pc_final_pred:", int(pred_pc_final[idx]))
        print("bd_final_pred:", int(pred_bd_final[idx]))

    with open(OUT_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("\n[info] saved:")
    print(" ", OUT_DIR / "y_pc_raw.npy")
    print(" ", OUT_DIR / "y_pc_final.npy")
    print(" ", OUT_DIR / "y_bd_final.npy")
    print(" ", OUT_DIR / "summary.json")


if __name__ == "__main__":
    main()

[info] using child model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx

[x_hw_all]
  shape: (50, 1, 140, 1)
  dtype: float32
  min  : -77.0
  max  : 59.0
  mean : 0.0024285715
  sample0: [[[ 59.]
  [ 11.]
  [-34.]
  [-66.]
  [-73.]
  [-55.]
  [-31.]
  [-18.]
  [-10.]
  [  5.]
  [ 14.]
  [ 13.]
  [ 13.]
  [ 13.]
  [ 12.]
  [ 11.]
  [ 12.]
  [ 10.]
  [ 10.]
  [  9.]
  [ 10.]
  [  8.]
  [  7.]
  [  5.]
  [  3.]
  [  3.]
  [  2.]
  [ -1.]
  [ -2.]
  [ -3.]
  [ -5.]
  [ -6.]
  [ -6.]
  [ -8.]
  [-11.]
  [ -9.]
  [-10.]
  [ -9.]
  [-11.]
  [ -9.]
  [ -7.]
  [ -7.]
  [ -5.]
  [ -3.]
  [ -1.]
  [  2.]
  [  3.]
  [  2.]
  [  3.]
  [  4.]
  [  5.]
  [  5.]
  [  6.]
  [  7.]
  [  7.]
  [  6.]
  [  7.]
  [  5.]
  [  6.]
  [  5.]
  [  6.]
  [  6.]
  [  6.]
  [ 10.]
  [  8.]
  [  8.]
  [ 10.]
  [ 10.]
  [  9.]
  [ 10.]
  [ 12.]
  [  8.]
  [  9.]
  [ 10.]
  [ 10.]
  [ 10.]
  [  7.]
  [  7.]
  [  7.]
  [  7.]
  [  5.]
 

/home/slowman/Desktop/project/Thesis/finn/src/finn/custom_op/fpgadataflow/rtl/streamingfifo_rtl.py:70: UserWarning: StreamingFIFO_rtl_4: rounding-up FIFO depth from 2240 to 4096 for impl_style=vivado
  warnings.warn(
/home/slowman/Desktop/project/Thesis/finn/src/finn/custom_op/fpgadataflow/rtl/streamingfifo_rtl.py:70: UserWarning: StreamingFIFO_rtl_5: rounding-up FIFO depth from 420 to 512 for impl_style=vivado
  warnings.warn(
/home/slowman/Desktop/project/Thesis/finn/src/finn/custom_op/fpgadataflow/rtl/streamingfifo_rtl.py:70: UserWarning: StreamingFIFO_rtl_9: rounding-up FIFO depth from 367 to 512 for impl_style=vivado
  warnings.warn(
/home/slowman/Desktop/project/Thesis/finn/src/finn/custom_op/fpgadataflow/rtl/streamingfifo_rtl.py:70: UserWarning: StreamingFIFO_rtl_10: rounding-up FIFO depth from 6720 to 8192 for impl_style=vivado
  warnings.warn(
/home/slowman/Desktop/project/Thesis/finn/src/finn/custom_op/fpgadataflow/rtl/streamingfifo_rtl.py:70: UserWarning: StreamingFIFO_rtl_2

[progress] pc child 10/50
[progress] pc child 20/50
[progress] pc child 30/50
[progress] pc child 40/50
[progress] pc child 50/50

[pc_raw]
  shape: (50, 1, 1, 5)
  dtype: float32
  min  : -1036012.0
  max  : 530692.0
  mean : -484645.6
  sample0: [[[ 430937. -735194. -681037. -818776. -629093.]]]
  sample1: [[[ 460439. -650723. -844528. -721681. -633197.]]]
  sample2: [[[ 232264. -489421. -751060. -596628. -825782.]]]

========== summary ==========
child_model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx
num_samples: 50
acc_pc_raw: 1.0
acc_bd_raw: 0.3
agree_raw: 0.3
acc_pc_final: 1.0
acc_bd_final: 0.0
agree_final: 0.0

========== diagnosis ==========
board raw 已经和 pc raw 不一致，问题在板子输出语义/板级执行，不在 sw_tail。

========== mismatch samples ==========
num raw mismatches: 35

--- sample 0 ---
label        : 0
pc_raw       : [ 430937. -735194. -681037. -818776. -629093.]
board_raw    : [1. 2. 3. 0. 3.]
pc_raw_pred 

In [4]:
#check minimize data width
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
MODEL_PATH = BUILD_DIR / "intermediate_models" / "step_set_fifo_depths.onnx"

m = ModelWrapper(str(MODEL_PATH))

print("model:", MODEL_PATH)

for n in reversed(m.graph.node):
    if n.domain == "finn.custom_op.fpgadataflow":
        print("\nnode:", n.name, n.op_type)
        print("output tensor:", n.output[0])

        try:
            print("tensor dtype anno:", m.get_tensor_datatype(n.output[0]))
        except Exception as e:
            print("tensor dtype anno: <error>", e)

        try:
            inst = getCustomOp(n)
            for attr in ["outputDataType", "accDataType", "PE", "SIMD", "MW", "MH"]:
                try:
                    print(attr, "=", inst.get_nodeattr(attr))
                except Exception:
                    pass
        except Exception as e:
            print("getCustomOp failed:", e)

        break

model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx
